In [ ]:
import os
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, Tool  
from agents.mcp import MCPServerStdio
from openai import AsyncOpenAI
from datetime import datetime, date
import asyncio

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

google_api_key = os.getenv('GOOGLE_API_KEY')
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
gemini_model = OpenAIChatCompletionsModel(model="gemini-3-flash-preview", openai_client=gemini_client)

openai_model = "gpt-5-mini"
project_path = os.path.abspath(os.path.join(os.getcwd()))

files_params = {
    "command": "npx",
    "args": [
        "-y",
        "@modelcontextprotocol/server-filesystem",
        project_path
    ]
}

file_server = MCPServerStdio(params=files_params,client_session_timeout_seconds=60)

instructions = """
You are a certified financial analyst and expert of SEC EDGAR filings.
You are given a financial statement XBRL concept and best match against a predifined income statement line 
item for a company. The file with the mapping is income_statement_xbrl_mapping.py. 
The file already has the mapping for the most common concepts.
Use MCP Server tools to access income_statement_xbrl_mapping.py"""

async def verify_concept_mapping(instructions) -> Agent:
    instructions = instructions
    verify_concept_mapping = Agent(
        name="Verify Concept Mapping Agent",
        instructions=instructions,
        mcp_servers=[file_server],
        model=openai_model
    )
    return verify_concept_mapping
    
concept_agent = await verify_concept_mapping(instructions)

concept = "SellingAndMarketingExpense"
user_instructions = f"""
You are given the XBR concept {concept}.
Map the given XBRL concept to the best match income statement category line item in the predifined mapping file
test_predifined_xbrl_mapping.py.
Return the income statement line item from the file_server
test_predifined_xbrl_mapping.py. Return on the line item without extra verbage.
If the concept is already mapped, just return "Already mapped" without any extra verbage.
"""

#START MCP SERVER
await file_server.connect()

#RUN AGENT
with trace("Verify Concept Mapping Agent"):
    result = await Runner.run(concept_agent, user_instructions)


CancelledError: 

In [ ]:
print(result.final_output)

In [3]:
import os
from edgar import Company , set_identity
from edgar.xbrl import XBRLS
from dotenv import load_dotenv

# Load environment variables
load_dotenv()


# Set SEC identity to avoid 403 blocks
sec_identity = os.getenv("SEC_ID")

set_identity(sec_identity)

company = Company("MSFT")

filing = company.latest("10-K")



/home/pedro/projects/fin_import2/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
type(filing)

edgar.entity.filings.EntityFiling

In [7]:
# THIS CODE DOWNLOADS FINACIAL STATEMENTS FROM SEC WITH XBRL CONCEPTS
# REMOVES THE DIMESIONS LINES (dimensions True)

from edgar import Company
from edgar import xbrl
from edgar.xbrl import XBRLS
import pandas as pd
import time

tickers_df = pd.read_csv("../data/test_tickers.csv")
tickers_list = tickers_df['tickers'].tolist()


In [8]:
all_cashflow_df = pd.DataFrame()
i = 0

for ticker in tickers_list:
    print("extracting cashflow_statement for", ticker)
    company = Company(ticker)
    filing = company.get_filings(form="10-K", year = "2024", amendments=False).latest(1)
    print(i)
    try:
        xbrl = filing.xbrl()
        cashflow = xbrl.statements.cashflow_statement()
    except Exception as e:
        print(f"Error extracting XBRL for {ticker}: {e}. Skipping to next ticker.")
        continue
  

    cashflow = xbrl.statements.cashflow_statement()
    cashflow_df = cashflow.to_dataframe()
    
    ### EXTRACT DIMENSIONS FALSE
    cashflow_df = cashflow_df[cashflow_df['dimension'] != True]
    #cashflow_df = cashflow_df.iloc[:,:3]
    
    #BREAK AFTER 15 TICKERS FOR TESTING


    time.sleep(0.25)
    all_cashflow_df = pd.concat([all_cashflow_df, cashflow_df], ignore_index=True)
    #all_cashflow_df = cashflow_df
    i += 1
    if i >= 15:
        break 

all_cashflow_df.to_csv("all_cashflow_df_forone.csv")


extracting cashflow_statement for AAPL
0
extracting cashflow_statement for ABT
1
extracting cashflow_statement for ACN
2
extracting cashflow_statement for ADBE
3
extracting cashflow_statement for ADM
4
extracting cashflow_statement for AEO
5
extracting cashflow_statement for ALB
6
extracting cashflow_statement for ALL
7
extracting cashflow_statement for AMGN
8
extracting cashflow_statement for AMT
9
extracting cashflow_statement for ANF
10
extracting cashflow_statement for APD
11
extracting cashflow_statement for ARE
12
extracting cashflow_statement for AVGO
13
extracting cashflow_statement for BAC
14


In [9]:
all_cashflow_df.describe()

,2024-09-28,2023-09-30,2022-09-24,level,weight,preferred_sign,2023-12-31,2022-12-31,2021-12-31,2024-08-31,...,2022-12-02,2021-12-03,2024-02-03,2023-01-28,2022-01-29,2024-09-30,2022-09-30,2024-11-03,2023-10-29,2022-10-30
count,2.700000e+01,7.100000e+01,2.700000e+01,692.000000,530.000000,586.000000,3.230000e+02,3.230000e+02,3.230000e+02,2.900000e+01,...,3.200000e+01,3.200000e+01,6.000000e+01,5.900000e+01,5.900000e+01,4.400000e+01,4.400000e+01,3.300000e+01,3.300000e+01,3.300000e+01
mean,1.607704e+10,5.325779e+09,1.603233e+10,2.332370,0.150943,0.221843,4.238869e+09,2.310985e+09,6.124064e+09,8.618030e+08,...,6.186250e+08,6.068438e+08,5.268888e+07,3.867102e+06,2.240642e+07,7.625591e+08,3.892318e+08,4.166121e+09,1.364606e+09,1.313303e+09
std,4.236506e+10,2.426983e+10,4.313139e+10,0.914235,0.989476,0.975915,2.257767e+10,1.867363e+10,3.951323e+10,3.357515e+09,...,2.325007e+09,2.015287e+09,1.619268e+08,1.499152e+08,1.886918e+08,1.835769e+09,1.204947e+09,1.037851e+10,5.095799e+09,4.818202e+09
min,-1.219830e+11,-1.084880e+11,-1.107490e+11,1.000000,-1.000000,-1.000000,-4.439100e+10,-1.341900e+11,-3.132910e+11,-7.061818e+09,...,-6.825000e+09,-4.301000e+09,-3.265710e+08,-4.078930e+08,-5.946010e+08,-4.919200e+09,-3.857200e+09,-2.307000e+10,-1.562300e+10,-1.581600e+10
25%,1.811000e+09,3.000000e+05,1.954500e+09,2.000000,-1.000000,-1.000000,0.000000e+00,0.000000e+00,4.000000e+06,-8.672000e+06,...,5.250000e+06,1.500000e+06,-3.378000e+06,-1.850000e+07,-9.098500e+06,0.000000e+00,0.000000e+00,1.000000e+07,0.000000e+00,-3.000000e+06
50%,9.447000e+09,6.154000e+08,7.520000e+09,2.000000,1.000000,1.000000,1.120000e+08,1.150000e+08,2.040000e+08,1.449200e+08,...,1.620000e+08,1.870000e+08,2.257600e+07,7.862000e+06,1.306500e+07,5.940000e+07,4.730000e+07,5.930000e+08,2.280000e+08,2.530000e+08
75%,1.539300e+10,3.610600e+09,1.720700e+10,3.000000,1.000000,1.000000,1.090236e+09,1.259823e+09,1.998743e+09,1.941590e+09,...,5.727500e+08,8.515000e+08,1.000558e+08,5.516200e+07,8.744450e+07,1.384800e+09,5.909250e+08,5.741000e+09,1.773000e+09,1.455000e+09
max,1.182540e+11,1.105430e+11,1.221510e+11,5.000000,1.000000,1.000000,2.909590e+11,1.349620e+11,3.627360e+11,9.131027e+09,...,7.838000e+09,7.230000e+09,6.534220e+08,4.062960e+08,4.196290e+08,6.796700e+09,3.170600e+09,3.995400e+10,1.808500e+10,1.673600e+10


In [10]:
all_cashflow_df = pd.read_csv("all_cashflow_df.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'all_cashflow_df.csv'

In [ ]:
all_unique_cashflow_df = all_cashflow_df.drop_duplicates(subset=['concept']).reset_index(drop=True)
all_unique_cashflow_df.describe()

In [ ]:
concepts_list = all_unique_cashflow_df.iloc[:,1:2].values.tolist()


In [ ]:
#print(all_unique_inc_stmt_df.iloc[:,0])
#concepts_list = all_unique_cashflow_df.iloc[:,:1].values.tolist()

In [ ]:
## MAP CONCEPTS TO LINE ITEMS
from openai import AsyncOpenAI, OpenAI
import asyncio
import json

# CHANGE TO FINANCIAL STATEMENT YOU WANT TO MAP
from xbrl_mappings.cash_flow_xbrl_mapping import CASH_FLOW_MAPPING

# OLLAMA SETTINGS
# OLLAMA_BASE_URL = "http://172.17.112.1:11434/v1"
# ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
# ollama_model = "deepseek-r1:8b"

google_api_key = os.getenv('GOOGLE_API_KEY')
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

gemini_client = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
gemini_model = "gemini-3-flash-preview"

openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
openrouter_client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=openrouter_api_key)

## CHOOSE OPENROUTER MODEL
#openrouter_model = "stepfun/step-3.5-flash:free"
openrouter_model = "x-ai/grok-4.1-fast"
#openrouter_model = "openai/gpt-oss-120b"
#openrouter_model = "moonshotai/kimi-k2.5"

#### USE EXTRA BODY TO FORCE OPENROUTER TO USE CEREBRAS FOR GPT-OSS-120B
# openrouter_extra_body={
#     "provider": {
#         "only": ["cerebras"],        # restrict to Cerebras
#         #"only": ["deepinfra"],        # restrict to Cerebras
#         "allow_fallbacks": False,    # fail instead of switching providers
#     },
# }  

all_cashflow_df = pd.read_csv("all_cashflow_df.csv")
all_unique_cashflow_df = all_cashflow_df.drop_duplicates(subset=['concept']).reset_index(drop=True)
all_unique_cashflow_df.describe()

concepts_list = all_unique_cashflow_df.iloc[:,1:2].values.tolist()

cashflow_mapping_json = json.dumps(CASH_FLOW_MAPPING)

# import random

# # Select 15 random concepts from concepts_list
# random_concepts = random.sample(concepts_list, min(15, len(concepts_list)))

i = 0
grok_response = []
for concept in concepts_list:

    instructions = f""" You are a finance analyst and expert with SEC EDGAR filings and XBRL concepts.
    You are given a EDGAR financial statement XBRL concept and 
    a JSON file with predifined XBRL concepts to income statement line mappings. 
    Your job is to map the given XBRL concept to the best match income statement line item in the 
    predifined mapping file JSON. Return only the income statement line item that 
    best matches the given XBRL concept. No extra verbage.
    JSON file: {cashflow_mapping_json}
    XBRL concept: {concept}"""

    i += 1 
    
    response = openrouter_client.chat.completions.create(
        model=openrouter_model,
        messages=[{"role": "user", "content": instructions}],
        #NEED TO ADD EXTRA BODY TO FORCE OPENROUTER TO USE CEREBRAS FOR GPT-OSS-120B
        #extra_body=openrouter_extra_body
    )

    print(i," concept", concept[0], "response", response.choices[0].message.content)

    # Initialize oss_response on the first iteration and add each response to a list
    grok_response.append(response.choices[0].message.content)

    if i > 15:
        break


## CREATE A LIST OF DICTS WITH THE CONCEPT AND THE RESPONSE
grok_response_dicts = [
    {"concept": concept[0], "cashflow_mapping": response}
    for concept, response in zip(concepts_list, grok_response)
]


file_path = openrouter_model.rsplit('/', 1)[-1]

### SAVE TO FILE
# Convert oss_response (list) to newline-separated string and save to file
with open(f"{file_path}_response_dicts.txt", "w") as f:
    f.write("\n".join(str(item) for item in grok_response_dicts))

print(f"{file_path}_response_dicts.txt", "created")

In [44]:
### THIS CODE IS USED TO IMPORT THE CONCEPT RESPONSE DICTS INTO THE DUCKDB DATABASE
import duckdb
import ast

DB_PATH = "data/xbrl_mappings_multi.duckdb"
VALID_STATEMENT_TYPES = {"income", "balance", "cashflow"}

def import_grok_response_dicts(file_path: str, statement_type: str, db_path: str = DB_PATH) -> dict:
    """
    Import grok_response_dicts.txt into the ai_discovered_mappings table
    of data/xbrl_mappings_multi.duckdb.

    Each line in the file is a Python dict literal like:
        {'concept': 'PolicyFeeIncome', 'income_statement_mapping': 'revenue'}

    Args:
        file_path: Path to the response dicts text file.
        statement_type: One of 'income', 'balance', or 'cashflow'.

    Returns dict with counts: {'inserted': N, 'skipped': N, 'errors': N}
    """
    if statement_type not in VALID_STATEMENT_TYPES:
        raise ValueError(
            f"Invalid statement_type '{statement_type}'. "
            f"Must be one of: {', '.join(sorted(VALID_STATEMENT_TYPES))}"
        )

    con = duckdb.connect(db_path)
    inserted = 0
    skipped = 0
    errors = 0

    # Detect the mapping key from the first non-empty line
    mapping_key = None
    with open(file_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            entry = ast.literal_eval(line)
            # Find the key that isn't 'concept'
            mapping_key = [k for k in entry if k != "concept"][0]
            break

    if not mapping_key:
        con.close()
        raise ValueError(f"Could not detect mapping key from {file_path}")

    print(f"Using mapping key: '{mapping_key}', statement_type: '{statement_type}'")

    with open(file_path, "r") as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                entry = ast.literal_eval(line)
                concept = entry["concept"]
                field_name = entry[mapping_key]

                # Check if this mapping already exists
                existing = con.execute(
                    """SELECT id FROM ai_discovered_mappings
                       WHERE statement_type = ?
                         AND field_name = ?
                         AND concept = ?""",
                    [statement_type, field_name, concept],
                ).fetchone()

                if existing:
                    skipped += 1
                    print(f"  SKIP (duplicate): {concept} -> {field_name}")
                    continue

                con.execute(
                    """INSERT INTO ai_discovered_mappings
                       (statement_type, field_name, concept, confidence_score)
                       VALUES (?, ?, ?, 0.8)""",
                    [statement_type, field_name, concept],
                )
                inserted += 1
                print(f"  INSERT: {concept} -> {field_name}")

            except Exception as e:
                errors += 1
                print(f"  ERROR line {line_num}: {e}")

    con.close()
    summary = {"inserted": inserted, "skipped": skipped, "errors": errors}
    print(f"\nDone. {summary}")
    return summary

In [61]:
# --- Import the grok response dicts file --- VALID_STATEMENT_TYPES = {"income", "balance", "cashflow"}
result = import_grok_response_dicts("balance_st_response_dicts.txt", statement_type="balance")


Using mapping key: 'balance_mapping', statement_type: 'balance'
  INSERT: LicensedContentCostsAndAdvances -> other_noncurrent_assets
  INSERT: ContractWithCustomerAssetGross -> other_current_assets
  INSERT: RightToRecoverForCoveredLosses -> other_noncurrent_assets
  INSERT: NaturalGasCostOverRecoveryShortTerm -> other_current_liabilities
  INSERT: Fuel -> inventory
  INSERT: CustomerReturnsRebatesAndIncentives -> accrued_expenses
  INSERT: InformationTechnology -> other_noncurrent_assets
  INSERT: RevenueRemainingPerformanceObligation -> deferred_revenue_current
  INSERT: SecuredNotesAndLoansReceivableNet -> long_term_investments
  INSERT: LoansAndLeasesReceivableConsumerMortgageNetOfDeferredIncome -> accounts_receivable
  INSERT: SelfFundedReceivablesNet -> accounts_receivable
  INSERT: CashAndCashEquivalentsAtCarryingValueIncludingDiscontinuedOperations -> cash
  INSERT: RestrictedCashCurrent -> other_current_assets
  INSERT: RestrictedCashAndInvestments -> other_current_assets
  IN

In [46]:
def concept_has_mapping(concept: str, db_path: str = DB_PATH) -> dict | None:
    """
    Check whether a concept already has an entry in data/xbrl_mappings_multi.duckdb.
    Searches both core_concept_mappings and ai_discovered_mappings.

    Args:
        concept: The XBRL concept string, e.g. 'us-gaap_EarningsPerShareBasic'

    Returns:
        A dict with the mapping info if found, e.g.:
            {'source': 'core', 'statement_type': 'income', 'field_name': 'basic_eps', 'concept': '...'}
        or None if the concept has no entry.
    """
    con = duckdb.connect(db_path, read_only=True)

    # 1) Check core_concept_mappings first (higher trust)
    row = con.execute(
        """SELECT statement_type, field_name, concept, priority
           FROM core_concept_mappings
           WHERE concept = ?
           ORDER BY priority ASC
           LIMIT 1""",
        [concept],
    ).fetchone()

    if row:
        con.close()
        return {
            "source": "core",
            "statement_type": row[0],
            "field_name": row[1],
            "concept": row[2],
            "priority": row[3],
        }

    # 2) Check ai_discovered_mappings
    row = con.execute(
        """SELECT statement_type, field_name, concept, confidence_score
           FROM ai_discovered_mappings
           WHERE concept = ?
           LIMIT 1""",
        [concept],
    ).fetchone()

    con.close()

    if row:
        return {
            "source": "ai_discovered",
            "statement_type": row[0],
            "field_name": row[1],
            "concept": row[2],
            "confidence_score": row[3],
        }

    return None

In [63]:

# --- Test the lookup function ---
print("\n--- Lookup tests ---")

# Should find in ai_discovered after import
test1 = concept_has_mapping("IndefiniteLivedTrademarks")
print(f"IndefiniteLivedTrademarks: {test1}")

# Should find in core_concept_mappings (us-gaap standard concept)
test2 = concept_has_mapping("Fuel")
print(f"Fuel: {test2}")

# Should return None (doesn't exist)
test3 = concept_has_mapping("NonExistentConcept_XYZ123")
print(f"NonExistentConcept_XYZ123: {test3}")


--- Lookup tests ---
IndefiniteLivedTrademarks: {'source': 'ai_discovered', 'statement_type': 'balance', 'field_name': 'intangible_assets', 'concept': 'IndefiniteLivedTrademarks', 'confidence_score': 0.800000011920929}
Fuel: {'source': 'ai_discovered', 'statement_type': 'balance', 'field_name': 'inventory', 'concept': 'Fuel', 'confidence_score': 0.800000011920929}
NonExistentConcept_XYZ123: None


In [ ]:

async def verify_concept_mapping(instructions) -> Agent:
    instructions = instructions,
    verify_concept_mapping = Agent(
        name="Verify Concept Mapping Agent",
        instructions=instructions

In [ ]:

i = 0
for concept in concepts_list:
    while i < 15:
        print(i,concept[0])
        i += 1
   


In [ ]:
import pandas as pd
tickers_df = pd.read_csv("test_tickers.csv")

### REMOVE DUPLICATES
unique_tickers_df = tickers_df.drop_duplicates(subset=['tickers']).reset_index(drop=True)
unique_tickers_df.to_csv("test_tickers.csv")

In [ ]:
unique_tickers_df.describe()

In [ ]:
#unique_tickers_df.describe()

# Check if 'NVDA' is in the dataframe
'NFLX' in unique_tickers_df['tickers'].str.upper().values

In [ ]:
def get_income_dataframe(ticker:str):
    c = Company(ticker)
    filings = c.get_filings(form="10-K").latest(5)
    xbs = XBRLS.from_filings(filings)
    income_statement = xbs.statements.income_statement()
    income_df = income_statement.to_dataframe()
    return income_df

In [ ]:
inc_stmt = xbrl.statements.income_statement()

In [ ]:
inc_stmt_df = inc_stmt.to_dataframe
inc_stmt_df.to_csv("inc_stmt_df.csv")

In [ ]:
results = (xbrl.query()
            .by_concept("us-gaap:RevenueFromContractWithCustomerExcludingAssessedTax")
           )

In [69]:
"""Script to compare two text files row by row and report differences.

Suitable for use in a Jupyter notebook cell.
"""

import ast
from pathlib import Path
from typing import List, Tuple

import pandas as pd


def _parse_response_line(line: str) -> tuple[str, str]:
    """Parse a dict line into (concept, response). Returns (concept, mapping or raw line)."""
    if not line or line == "[MISSING]":
        return ("[MISSING]", line or "[MISSING]")
    try:
        d = ast.literal_eval(line)
        concept = d.get("concept", "")
        response = d.get("cashflow_mapping", str(d))
        return (concept or "[unknown]", response)
    except (ValueError, SyntaxError):
        return ("[parse error]", line)


def _differences_to_dataframe(
    differences: List[Tuple[int, str, str]],
) -> pd.DataFrame:
    """Build a DataFrame with columns Concept, response from file1, response from file2."""
    rows = []
    for _line_num, content1, content2 in differences:
        concept1, response1 = _parse_response_line(content1)
        concept2, response2 = _parse_response_line(content2)
        concept = concept1 if concept1 != "[MISSING]" else concept2
        rows.append({
            "Concept": concept,
            "response from file1": response1,
            "response from file2": response2,
        })
    return pd.DataFrame(rows)


def compare_files(
    file1_path: str | Path,
    file2_path: str | Path,
) -> pd.DataFrame:
    """Compare two text files line by line and print differences.

    Returns a DataFrame with columns: Concept, response from file1, response from file2.
    """
    file1_path = Path("gemini_response_dicts.txt")
    file2_path = Path("arcee-ai_response_dicts.txt")

    # Read both files
    with file1_path.open('r', encoding='utf-8') as f:
        lines1 = [line.rstrip('\n\r') for line in f]

    with file2_path.open('r', encoding='utf-8') as f:
        lines2 = [line.rstrip('\n\r') for line in f]

    # Print summary
    print(f"File 1: {file1_path.name} ({len(lines1)} lines)")
    print(f"File 2: {file2_path.name} ({len(lines2)} lines)")
    print("-" * 80)

    # Find differences
    max_lines = max(len(lines1), len(lines2))
    differences: List[Tuple[int, str, str]] = []

    for line_num in range(max_lines):
        line1 = lines1[line_num] if line_num < len(lines1) else None
        line2 = lines2[line_num] if line_num < len(lines2) else None

        if line1 != line2:
            differences.append((line_num + 1, line1 or "[MISSING]", line2 or "[MISSING]"))

    # Print results and build DataFrame
    if not differences:
        print("✓ Files are identical!")
        diff_df = pd.DataFrame(columns=["Concept", "response from file1", "response from file2"])
    else:
        print(f"Found {len(differences)} difference(s):\n")
        # for line_num, content1, content2 in differences:
        #     print(f"Line {line_num}:")
        #     print(f"  File 1: {content1}")
        #     print(f"  File 2: {content2}")
        #     print()
        diff_df = _differences_to_dataframe(differences)

    return diff_df


def remove_diff_concepts_and_save(
    diff_df: pd.DataFrame,
    file1_path: str | Path,
    file2_path: str | Path,
    suffix: str = "_filtered",
) -> tuple[Path, Path]:
    """Remove concepts listed in diff_df from file1 and file2; save under new names.

    Writes file1 to {stem}{suffix}{suffix_ext} and same for file2 (e.g. name_filtered.txt).
    Returns the paths to the saved files.

    Args:
        diff_df: DataFrame with a 'Concept' column (concepts to remove).
        file1_path: Path to first file.
        file2_path: Path to second file.
        suffix: Suffix for the new filenames (default '_filtered').

    Returns:
        (path_to_saved_file1, path_to_saved_file2)
    """
    file1_path = Path(file1_path)
    file2_path = Path(file2_path)
    if diff_df.empty or "Concept" not in diff_df.columns:
        raise ValueError("diff_df must be non-empty and have a 'Concept' column")
    concepts_to_remove = set(diff_df["Concept"].dropna().astype(str))

    def _filter_lines(path: Path) -> list[str]:
        with path.open("r", encoding="utf-8") as f:
            lines = [line.rstrip("\n\r") for line in f]
        kept = [
            line
            for line in lines
            if _parse_response_line(line)[0] not in concepts_to_remove
        ]
        return kept

    def _output_path(path: Path) -> Path:
        return path.parent / f"{path.stem}{suffix}{path.suffix}"

    out1 = _output_path(file1_path)
    out2 = _output_path(file2_path)

    for path, out_path in [(file1_path, out1), (file2_path, out2)]:
        kept = _filter_lines(path)
        with out_path.open("w", encoding="utf-8") as f:
            f.write("\n".join(kept))
            if kept:
                f.write("\n")
    return (out1, out2)


# Main execution for notebook use
# if __name__ == "__main__":
#     file1 = Path("/home/pedro/projects/fin_import2/gemini_response_dicts.txt")
#     file2 = Path("/home/pedro/projects/fin_import2/arcee-ai_response_dicts.txt")
#     diff_df = compare_files(file1, file2)

In [70]:
diff_df = compare_files(Path("gemini_response_dicts.txt"), Path("arcee-ai_response_dicts.txt"))
# out1, out2 = remove_diff_concepts_and_save(diff_df, Path("gemini_response_dicts.txt"), Path("arcee-ai_response_dicts.txt"))
# # Optional: use a different suffix
# out1, out2 = remove_diff_concepts_and_save(diff_df, file1, file2, suffix="_without_diffs")

File 1: gemini_response_dicts.txt (3096 lines)
File 2: arcee-ai_response_dicts.txt (3096 lines)
--------------------------------------------------------------------------------
Found 1387 difference(s):



In [71]:
diff_df.to_csv("diff_df.csv")

In [72]:
diff_df.describe()

,Concept,response from file1,response from file2
count,1387,1387,1387
unique,1387,29,54
top,us-gaap_IncreaseDecreaseInOtherReceivables,other_operating_activities,other_investing_activities
freq,1,579,242


In [94]:
eval_df = pd.read_csv("evaluated_mappings.csv")

eval_df.drop(columns=['Unnamed: 0'], inplace=True)

eval_list = eval_df.to_dict(orient="records")





In [95]:
print(eval_list)

[{'concept': 'us-gaap_IncreaseDecreaseInOtherReceivables', 'cashflow_mapping': 'change_accounts_receivable'}, {'concept': 'us-gaap_PaymentsRelatedToTaxWithholdingForShareBasedCompensation', 'cashflow_mapping': 'stock_based_compensation'}, {'concept': 'us-gaap_ProceedsFromRepaymentsOfCommercialPaper', 'cashflow_mapping': 'other_operating_activities'}, {'concept': 'us-gaap_SupplementalCashFlowInformationAbstract', 'cashflow_mapping': 'other_operating_activities'}, {'concept': 'abt_InvestingAndFinancingGainsLossesNet', 'cashflow_mapping': 'other_operating_activities'}, {'concept': 'us-gaap_IncreaseDecreaseInPrepaidDeferredExpenseAndOtherAssets', 'cashflow_mapping': 'other_operating_activities'}, {'concept': 'us-gaap_ProceedsFromSalesOfBusinessAffiliateAndProductiveAssets', 'cashflow_mapping': 'other_operating_activities'}, {'concept': 'us-gaap_ProceedsFromSaleOfOtherInvestments', 'cashflow_mapping': 'sales_investments'}, {'concept': 'us-gaap_IncreaseDecreaseInDeferredRevenue', 'cashflow_m

In [98]:
with open("eval_list.txt", "w") as f:
    for item in eval_list:
        f.write(str(item) + "\n")

In [51]:
#### THIS CODE IMPORTS MAPPINGS PREVIOUSLY EXPORTED FROM THE DATABASE
### AND CREATES A LIST OF DICTIONARIES
with open("balance_concept_mappings.txt", "r") as f:
    balance_concept_mappings = f.read()

import ast

# Parse the file content (it's a list of dictionaries)
balance_concept_mappings_list = ast.literal_eval(balance_concept_mappings)

# Extract ai_discovered_concepts from each dictionary in the list
ai_discovered_list = [
    {"concept": concept, "field_name": item["field_name"]}
    for item in balance_concept_mappings_list
    if "ai_discovered_concepts" in item
    for concept in (
        item["ai_discovered_concepts"]
        if isinstance(item["ai_discovered_concepts"], list)
        else [item["ai_discovered_concepts"]]
    )
]

In [52]:
print(ai_discovered_list)

[{'concept': 'dis_LicensedContentCostsAndAdvances', 'field_name': ''}, {'concept': 'us-gaap_ContractWithCustomerAssetGross', 'field_name': ''}, {'concept': 'v_RightToRecoverForCoveredLosses', 'field_name': ''}, {'concept': 'so_NaturalGasCostOverRecoveryShortTerm', 'field_name': ''}, {'concept': 'pseg_Fuel', 'field_name': ''}, {'concept': 'rok_CustomerReturnsRebatesAndIncentives', 'field_name': ''}, {'concept': 'fdx_InformationTechnology', 'field_name': '".None'}, {'concept': 'us-gaap_RevenueRemainingPerformanceObligation', 'field_name': '".Revenue'}, {'concept': 'vtr_SecuredNotesAndLoansReceivableNet', 'field_name': '".accounts_receivable'}, {'concept': 'cma_LoansAndLeasesReceivableConsumerMortgageNetOfDeferredIncome', 'field_name': '".accounts_receivable'}, {'concept': 'elv_SelfFundedReceivablesNet', 'field_name': '".accounts_receivable'}, {'concept': 'us-gaap_CashAndCashEquivalentsAtCarryingValueIncludingDiscontinuedOperations', 'field_name': '".cash'}, {'concept': 'us-gaap_Restricte

In [53]:
concepts_list = [d["concept"] for d in ai_discovered_list]

In [54]:
print(concepts_list)

['dis_LicensedContentCostsAndAdvances', 'us-gaap_ContractWithCustomerAssetGross', 'v_RightToRecoverForCoveredLosses', 'so_NaturalGasCostOverRecoveryShortTerm', 'pseg_Fuel', 'rok_CustomerReturnsRebatesAndIncentives', 'fdx_InformationTechnology', 'us-gaap_RevenueRemainingPerformanceObligation', 'vtr_SecuredNotesAndLoansReceivableNet', 'cma_LoansAndLeasesReceivableConsumerMortgageNetOfDeferredIncome', 'elv_SelfFundedReceivablesNet', 'us-gaap_CashAndCashEquivalentsAtCarryingValueIncludingDiscontinuedOperations', 'us-gaap_RestrictedCashCurrent', 'us-gaap_RestrictedCashAndInvestments', 'us-gaap_CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalentsIncludingDisposalGroupAndDiscontinuedOperations', 'us-gaap_CommonStockOtherValueOutstanding', 'mtb_CommonStockIssuable', 'us-gaap_OtherIndefiniteLivedIntangibleAssets', 'us-gaap_OtherIntangibleAssetsNet', 'us-gaap_IndefiniteLivedTrademarks', 'us-gaap_IndefiniteLivedContractualRights', 'us-gaap_InventoryRawMaterials', 'us-gaap_NuclearFuelNetO

In [55]:
concepts_list = [concept for concept in concepts_list if "Abstract" not in concept]

In [56]:
concepts_list = [concept.split('_', 1)[1] if '_' in concept else concept for concept in concepts_list]

In [57]:
print(concepts_list)

['LicensedContentCostsAndAdvances', 'ContractWithCustomerAssetGross', 'RightToRecoverForCoveredLosses', 'NaturalGasCostOverRecoveryShortTerm', 'Fuel', 'CustomerReturnsRebatesAndIncentives', 'InformationTechnology', 'RevenueRemainingPerformanceObligation', 'SecuredNotesAndLoansReceivableNet', 'LoansAndLeasesReceivableConsumerMortgageNetOfDeferredIncome', 'SelfFundedReceivablesNet', 'CashAndCashEquivalentsAtCarryingValueIncludingDiscontinuedOperations', 'RestrictedCashCurrent', 'RestrictedCashAndInvestments', 'CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalentsIncludingDisposalGroupAndDiscontinuedOperations', 'CommonStockOtherValueOutstanding', 'CommonStockIssuable', 'OtherIndefiniteLivedIntangibleAssets', 'OtherIntangibleAssetsNet', 'IndefiniteLivedTrademarks', 'IndefiniteLivedContractualRights', 'InventoryRawMaterials', 'NuclearFuelNetOfAmortization', 'OtherInventoryRawMaterialsNetofReserves', 'RetailRelatedInventoryMerchandise', 'InventoryHomesunderConstructionandFinishedHom

In [58]:
print(concepts_list[2])

RightToRecoverForCoveredLosses


In [59]:
abstract_count = sum(1 for concept in concepts_list if "Abstract" in concept)
print(f"Number of concepts with 'Abstract' in the name: {abstract_count}")

Number of concepts with 'Abstract' in the name: 0


In [60]:
len(concepts_list)

1298

In [31]:
i =0 
while i < 5:
    for concept in concept_list:
        print(f"{concept}")
        i += 1

us-gaap_AntidilutiveSecuritiesExcludedFromComputationOfEarningsPerShareAmount
crh_EarningsPerShareOtherDisclosureAbstract
us-gaap_NetIncomeLossPerOutstandingLimitedPartnershipUnitBasicNetOfTax
us-gaap_NetIncomeLossPerOutstandingLimitedPartnershipAndGeneralPartnershipUnitBasicAbstract
bk_EarningsPerShareBasicAndDilutedEPSAbstract
us-gaap_EarningsPerShareBasicAbstract
us-gaap_IncomeLossFromContinuingOperationsPerBasicShare
us-gaap_EarningsPerShareBasicOtherDisclosuresAbstract
us-gaap_ParticipatingSecuritiesDistributedAndUndistributedEarningsLossBasic
us-gaap_EarningsPerShareAbstract
ge_NetEarningsPerShareAbstract
cof_ParticipatingSecuritiesDistributedandUndistributedEarningsLossexcludingPreferredStockDividendsBasic
us-gaap_EarningsPerUnitAbstract
us-gaap_SupplementalIncomeStatementElementsAbstract
gehc_IncomeLossPerShareFromContinuingOperationsAbstract
has_EarningsPerShareBasicAndDilutedEPSAbstract
hrl_EarningsPerShareBasicAndDilutedEPSAbstract
ccl_EarningsPerShareBasicAndDilutedEPSAbstr